In [0]:
%sql
-- Create the app's detailed sales view
CREATE OR REPLACE VIEW workspace.portfolio_gold.sales_details AS
SELECT
    order_id,
    line_id,
    order_date,
    product,
    category,
    quantity,
    unit_price,
    line_revenue
FROM workspace.portfolio_silver.sales_clean
-- A view is a saved query. This gives the app a Gold entry point to the clean sales record without copying them

In [0]:
%sql
-- Create daily sales summaries
CREATE OR REPLACE TABLE workspace.portfolio_gold.daily_sales
USING DELTA
AS
SELECT
  order_date,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(quantity) AS units_sold,
  SUM(line_revenue) AS revenue,
  CAST(
    SUM(line_revenue) / COUNT(DISTINCT order_id)
    AS DECIMAL(18, 2)
  ) AS average_order_value
FROM workspace.portfolio_gold.sales_details
GROUP BY order_date;
-- We use COUNT(DISTINCT order_id) because an order can contain multiple product lines. Counting rows would overcount orders.

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Create product summaries by day
CREATE OR REPLACE TABLE workspace.portfolio_gold.daily_product_sales
USING DELTA
AS
SELECT
  order_date,
  category,
  product,
  SUM(quantity) AS units_sold,
  SUM(line_revenue) AS revenue
FROM workspace.portfolio_gold.sales_details
GROUP BY
  order_date,
  category,
  product;

-- Keeping the date and category allows the app to filter these summaries later.

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify the daily results
SELECT *
FROM workspace.portfolio_gold.daily_sales
ORDER BY order_date;

order_date,total_orders,units_sold,revenue,average_order_value
2026-09-01,2,4,295.00,147.50
2026-09-02,2,13,30.00,15.00
2026-09-03,2,2,145.00,72.50


In [0]:
%sql
-- Then check the overall totals
SELECT
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(quantity) AS units_sold,
  SUM(line_revenue) AS total_revenue
FROM workspace.portfolio_gold.sales_details;

total_orders,units_sold,total_revenue
6,19,470.00
